In [20]:
import pandas as pd

df = pd.read_csv('Soil_Data.csv')

def find_col(df, keywords):
    for col in df.columns:
        if any(kw.lower() in col.lower() for kw in keywords):
            return col
    return None

ph_col = find_col(df, ['ph'])
ec_col = find_col(df, ['e.c.', 'ec'])
oc_col = find_col(df, ['organic', 'oc'])
p_col  = find_col(df, ['p2o5', 'phosphorus'])
k_col  = find_col(df, ['k2o', 'potassium'])
zn_col = find_col(df, ['zn', 'zinc'])
fe_col = find_col(df, ['fe', 'iron'])
cu_col = find_col(df, ['cu', 'copper'])
mn_col = find_col(df, ['mn', 'manganese'])

def diagnose_soil(row):
    actions = []

    # pH Diagnosis
    if ph_col and pd.notna(row[ph_col]):
        val = row[ph_col]
        if val < 7.0:
            actions.append("pH is Low (Acidic) -> Apply Lime / Gypsum treatment")
        elif val > 8.50:
            actions.append("pH is High (Alkali) -> Apply Gypsum / Organic manure to lower alkalinity")

    # E.C. Diagnosis
    if ec_col and pd.notna(row[ec_col]):
        val = row[ec_col]
        if val >= 1.5:
            actions.append("E.C. is High/Critical -> Improve drainage & avoid saline water")

    # Organic Carbon
    if oc_col and pd.notna(row[oc_col]):
        val = row[oc_col]
        if val < 0.50:
            actions.append("Organic Carbon is Low -> Increase FYM (Farmyard Manure) / Compost")

    # Phosphorus (P2O5)
    if p_col and pd.notna(row[p_col]):
        val = row[p_col]
        if val < 23:
            actions.append("Phosphorus is Low -> Increase DAP / Single Super Phosphate (SSP)")

    # Potassium (K2O)
    if k_col and pd.notna(row[k_col]):
        val = row[k_col]
        if val < 144:
            actions.append("Potassium is Low -> Increase MOP (Muriate of Potash)")

    # Micronutrients
    if zn_col and pd.notna(row[zn_col]) and row[zn_col] < 0.6:
        actions.append("Zinc (Zn) is Low -> Apply Zinc Sulfate")

    if fe_col and pd.notna(row[fe_col]) and row[fe_col] < 4.5:
        actions.append("Iron (Fe) is Low -> Apply Ferrous Sulfate")

    if cu_col and pd.notna(row[cu_col]) and row[cu_col] < 0.2:
        actions.append("Copper (Cu) is Low -> Apply Copper Sulfate")

    if mn_col and pd.notna(row[mn_col]) and row[mn_col] < 2.0:
        actions.append("Manganese (Mn) is Low -> Apply Manganese Sulfate")

    if not actions:
        return "All measured soil parameters are in optimal range!"

    return " | ".join(actions)

df['Required_Actions'] = df.apply(diagnose_soil, axis=1)

df.to_csv('Soil_Data_Diagnosed.csv', index=False)

print("================ DIAGNOSTIC ENGINE COMPLETED ================")
print(f"Processed {len(df)} samples.")
print("\nSample Recommendations Generated:")
for i, rec in enumerate(df['Required_Actions'].head(5), 1):
    print(f"\nSample {i}:")
    print(rec)
print("=============================================================")

================ DIAGNOSTIC ENGINE COMPLETED ================
Processed 4656 samples.

Sample Recommendations Generated:

Sample 1:
Organic Carbon is Low -> Increase FYM (Farmyard Manure) / Compost | Zinc (Zn) is Low -> Apply Zinc Sulfate | Iron (Fe) is Low -> Apply Ferrous Sulfate

Sample 2:
Organic Carbon is Low -> Increase FYM (Farmyard Manure) / Compost | Zinc (Zn) is Low -> Apply Zinc Sulfate | Iron (Fe) is Low -> Apply Ferrous Sulfate

Sample 3:
pH is High (Alkali) -> Apply Gypsum / Organic manure to lower alkalinity | E.C. is High/Critical -> Improve drainage & avoid saline water | Organic Carbon is Low -> Increase FYM (Farmyard Manure) / Compost | Zinc (Zn) is Low -> Apply Zinc Sulfate | Iron (Fe) is Low -> Apply Ferrous Sulfate

Sample 4:
Organic Carbon is Low -> Increase FYM (Farmyard Manure) / Compost | Zinc (Zn) is Low -> Apply Zinc Sulfate | Iron (Fe) is Low -> Apply Ferrous Sulfate

Sample 5:
pH is High (Alkali) -> Apply Gypsum / Organic manure to lower alkalinity | Organ

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

df = pd.read_csv('Soil_Data_Diagnosed.csv')

def find_col(df, keywords):
    for col in df.columns:
        if any(kw.lower() in col.lower() for kw in keywords):
            return col
    return None

feature_cols = [
    find_col(df, ['ph']),
    find_col(df, ['e.c.', 'ec']),
    find_col(df, ['organic', 'oc']),
    find_col(df, ['p2o5', 'phosphorus']),
    find_col(df, ['k2o', 'potassium']),
    find_col(df, ['zn', 'zinc']),
    find_col(df, ['fe', 'iron']),
    find_col(df, ['cu', 'copper']),
    find_col(df, ['mn', 'manganese'])
]

ml_data = df.dropna(subset=feature_cols).copy()

X = ml_data[feature_cols]


y = ml_data['Required_Actions']

# 2. Train / Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 4. Model Evaluation
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("================ MODEL TRAINING COMPLETE ================")
print(f"Total Dataset Samples: {len(ml_data)}")
print(f"Training Set Size: {len(X_train)} rows")
print(f"Testing Set Size: {len(X_test)} rows")
print(f"\nModel Accuracy: {accuracy * 100:.2f}%")
print("\nDetailed Performance Report:")
print(classification_report(y_test, y_pred))
print("=========================================================")

# 5. Save Model and Feature Names
joblib.dump(rf_model, 'soil_classifier_model.pkl')
joblib.dump(feature_cols, 'feature_names.pkl')
print("\nSaved model as 'soil_classifier_model.pkl'!")

================ MODEL TRAINING COMPLETE ================
Total Dataset Samples: 4645
Training Set Size: 3716 rows
Testing Set Size: 929 rows

Model Accuracy: 99.35%

Detailed Performance Report:
                                                                                                                                                                                                                                                                                                                                                                       precision    recall  f1-score   support

                                                                           E.C. is High/Critical -> Improve drainage & avoid saline water | Organic Carbon is Low -> Increase FYM (Farmyard Manure) / Compost | Phosphorus is Low -> Increase DAP / Single Super Phosphate (SSP) | Zinc (Zn) is Low -> Apply Zinc Sulfate | Iron (Fe) is Low -> Apply Ferrous Sulfate       1.00      1.00      1.00         1
     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
